In [67]:
print("📚 Training pipeline")
results = []

for params in selected_params:
    #Tuple unpacking
    (
        n_estimators,
        max_depth,
        min_samples_leaf,
        min_samples_split,
        max_features
    ) = params
    
    # pipeline
    model = ImbPipeline([
        (
            "smote", SMOTE(random_state=42) 
        ),
        (
            "model", RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                min_samples_split=min_samples_split,
                max_features=max_features,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    #Fitting on the pipeline
    model.fit(X_train_temporal, y_train_temporal)
    
    # Validation
    val_probs = model.predict_proba(X_val_temporal)[:, 1]
    val_preds = model.predict(X_val_temporal)
    
    score = roc_auc_score(y_val_temporal, val_probs)
    pr_auc = average_precision_score(y_val_temporal, val_probs)
    recall = recall_score(y_val_temporal, val_preds)
    f1 = f1_score(y_val_temporal, val_preds)

    results.append(
        {
            "n_estimators": n_estimators,
            "max_depth": max_depth, 
            "min_samples_leaf": min_samples_leaf,
            "min_samples_split": min_samples_split,
            "max_features": max_features,
            "roc_auc": score,
            "pr_auc": pr_auc,
            "recall": recall,
            "f1": f1,
        }
    )

temporal_results = (
    pd.DataFrame(results).sort_values("roc_auc", ascending=False)
)
print(temporal_results.head())

📚 Training pipeline
    n_estimators  max_depth  min_samples_leaf  min_samples_split max_features  \
5            300       15.0                 3                 25         sqrt   
27           600       15.0                14                  5         sqrt   
12           700       25.0                10                 25         sqrt   
3            600       25.0                13                 50         sqrt   
15           500       30.0                 2                 50         sqrt   

     roc_auc    pr_auc    recall        f1  
5   0.980996  0.933280  0.888889  0.886918  
27  0.980778  0.932801  0.890370  0.878013  
12  0.980708  0.933237  0.887407  0.887407  
3   0.980440  0.932603  0.888889  0.880411  
15  0.980438  0.932187  0.878519  0.903275  


In [68]:
from datetime import datetime
print("Models' validation from all 3 experiments(2a, 2b, 2c)\n")
param_sets = [
    {
        "name": "Experiment 2A",
        "n_estimators": 450,
        "max_depth": 18,
        "min_samples_leaf": 1,
        "min_samples_split": 10,
        "max_features": "sqrt",
    },
    {
        "name": "Experiment 2B",
        "n_estimators": 568,
        "max_depth": 20,
        "min_samples_leaf": 2,
        "min_samples_split": 12,
        "max_features": 0.3,
    },
    {
        "name": "Experiment 2C",
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_leaf": 3,
        "min_samples_split": 25,
        "max_features": "sqrt",
    },
]

results = []
for params in param_sets:
    model = ImbPipeline(
        [
            ("smote", SMOTE(random_state=42)),
            ("model", RandomForestClassifier(
                n_estimators=params["n_estimators"],
                max_depth=params["max_depth"],
                min_samples_leaf=params["min_samples_leaf"],
                min_samples_split=params["min_samples_split"],
                max_features=params["max_features"],
                random_state=42,
                n_jobs=-1,
            ))
        ]
    )

    model.fit(
        X_train_temporal,
        y_train_temporal
    )

    probs = model.predict_proba(X_val_temporal)[:, 1]
    preds = model.predict(X_val_temporal)
    results.append({
        "name": params["name"],
        "ROC-AUC": roc_auc_score(y_val_temporal, probs),
        "PR-AUC": average_precision_score(y_val_temporal, probs),
        "Recall": recall_score(y_val_temporal, preds),
        "F1": f1_score(y_val_temporal, preds),
        "Confusion Matrix": confusion_matrix(y_val_temporal, preds),
    })

for result in results:
    print(f"\n{result['name']}")
    print(f"ROC-AUC: {result['ROC-AUC']:.4f}")
    print(f"PR-AUC : {result['PR-AUC']:.4f}")
    print(f"Recall : {result['Recall']:.4f}")
    print(f"F1     : {result['F1']:.4f}")
    print("Confusion Matrix:")
    print(result["Confusion Matrix"])

Models' validation from all 3 experiments(2a, 2b, 2c)


Experiment 2A
ROC-AUC: 0.9811
PR-AUC : 0.9329
Recall : 0.8726
F1     : 0.8938
Confusion Matrix:
[[7100   54]
 [  86  589]]

Experiment 2B
ROC-AUC: 0.9801
PR-AUC : 0.9336
Recall : 0.8785
F1     : 0.8992
Confusion Matrix:
[[7103   51]
 [  82  593]]

Experiment 2C
ROC-AUC: 0.9810
PR-AUC : 0.9333
Recall : 0.8889
F1     : 0.8869
Confusion Matrix:
[[7076   78]
 [  75  600]]


In [69]:
print("Models' base evaluation from all 3 experiments(2a, 2b, 2c)\n")

exp2a_model = joblib.load("models/random_forest/random_forest_exp2a.joblib")
exp2b_model = joblib.load("models/random_forest/random_forest_exp2b.joblib")
exp2c_model = joblib.load("models/random_forest/random_forest_exp2c_broadsearch.joblib")

model_list = [exp2a_model, exp2b_model, exp2c_model]
model_dict = {0: "⚖️ Model 2A evaluation\n", 1: "⚖️ Model 2B evaluation\n", 2: "⚖️ Model 2C evaluation\n"}
counter = 0
eval_results = []

for model in model_list:
    
    model_probs = model.predict_proba(X_test_temporal)[:, 1]
    model_preds = model.predict(X_test_temporal)
    
    probs = model.predict_proba(X_test_temporal)[:, 1]
    preds = model.predict(X_test_temporal)
    eval_results.append({
        "name": model_dict[counter],
        "ROC-AUC": roc_auc_score(y_test_temporal, probs),
        "PR-AUC": average_precision_score(y_test_temporal, probs),
        "Recall": recall_score(y_test_temporal, preds),
        "F1": f1_score(y_test_temporal, preds),
        "Confusion Matrix": confusion_matrix(y_test_temporal, preds),
    })

    counter += 1

for result in eval_results:
    print(f"\n{result['name']}")
    print(f"ROC-AUC: {result['ROC-AUC']:.4f}")
    print(f"PR-AUC : {result['PR-AUC']:.4f}")
    print(f"Recall : {result['Recall']:.4f}")
    print(f"F1     : {result['F1']:.4f}")
    print("Confusion Matrix:")
    print(result["Confusion Matrix"])

Models' base evaluation from all 3 experiments(2a, 2b, 2c)


⚖️ Model 2A evaluation

ROC-AUC: 0.9494
PR-AUC : 0.6713
Recall : 0.4755
F1     : 0.6268
Confusion Matrix:
[[8416   17]
 [ 214  194]]

⚖️ Model 2B evaluation

ROC-AUC: 0.9595
PR-AUC : 0.7047
Recall : 0.4779
F1     : 0.6331
Confusion Matrix:
[[8420   13]
 [ 213  195]]

⚖️ Model 2C evaluation

ROC-AUC: 0.9475
PR-AUC : 0.6639
Recall : 0.4755
F1     : 0.6228
Confusion Matrix:
[[8412   21]
 [ 214  194]]


In [70]:
# print("Model training with broad search hyperparameters for temporal split data")
# rf_exp2c_model = ImbPipeline([
#     (
#         "smote", SMOTE(random_state=42)
#     ),
#     (
#         "model", RandomForestClassifier(
#             n_estimators=300,
#             max_depth=15,
#             min_samples_leaf=3,
#             min_samples_split=25,
#             max_features="sqrt",
#             random_state=42,
#             n_jobs=-1
#         )
#     )
# ])

# # Training on full historical training period:
# rf_exp2c_model.fit(
#      train_period_df[feature_columns],
#      train_period_df["is_illicit"]
# )

# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# joblib.dump(rf_exp2c_model, f"models/random_forest/random_forest_exp2с_broadsearch{timestamp}.joblib")

# exp2c_probs = rf_exp2c_model.predict_proba(X_test_temporal)[:, 1]
# exp2c_preds = rf_exp2c_model.predict(X_test_temporal)

# print("ROC-AUC:", roc_auc_score(y_test_temporal, exp2c_probs))
# print("PR-AUC :", average_precision_score(y_test_temporal, exp2c_probs))
# print("Recall :", recall_score(y_test_temporal, exp2c_preds))
# print("F1     :", f1_score(y_test_temporal, exp2c_preds))
# print("Confusion matrix:")
# cm = confusion_matrix(y_test_temporal, exp2c_preds)
# print(cm)

In [71]:
  print("Retraining Experiment 2B config on time steps 1–41...")
  exp2b_full_model = ImbPipeline([
      ("smote", SMOTE(random_state=42)),
      ("model", RandomForestClassifier(
          n_estimators=568, max_depth=20, min_samples_leaf=2,
          min_samples_split=12, max_features=0.3,
          random_state=42, n_jobs=-1,
      ))
  ])
  exp2b_full_model.fit(X_train_temporal_full, y_train_temporal_full)
  joblib.dump(exp2b_full_model, "models/random_forest/random_forest_exp2b_full.joblib")

Retraining Experiment 2B config on time steps 1–41...


['models/random_forest/random_forest_exp2b_full.joblib']